## Replicando o tratamento de dados do primeiro, segundo e terceiro arquivo.

In [1]:
# Importando o pandas
import pandas as pd

In [2]:
# Visualizando a base de treino
treino = pd.read_csv('train.csv')
treino.head(3)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


In [3]:
# Visualizando a base de teste
teste = pd.read_csv('test.csv')
teste.head(3)

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q


In [4]:
# Eliminando as colunas com elevada cardinalidade
treino = treino.drop(['Name','Ticket','Cabin'],axis=1)
teste = teste.drop(['Name','Ticket','Cabin'],axis=1)

In [5]:
# Usando a média para substituir valores nulos na coluna de idade
treino.loc[treino.Age.isnull(),'Age'] = treino.Age.mean()
teste.loc[teste.Age.isnull(),'Age'] = teste.Age.mean()

In [6]:
# Tratando a coluna Embarked da base de treino usando a moda 
treino.loc[treino.Embarked.isnull(),'Embarked'] = treino.Embarked.mode()[0]

In [7]:
# E também a coluna Fare da base de teste usando a média
teste.loc[teste.Fare.isnull(),'Fare'] = teste.Fare.mean()

In [8]:
# Usando uma lambda function para tratar a coluna "Sex"
treino['MaleCheck'] = treino.Sex.apply(lambda x: 1 if x == 'male' else 0)
teste['MaleCheck'] = teste.Sex.apply(lambda x: 1 if x == 'male' else 0)

In [9]:
# Eliminando a coluna Sex
treino = treino.drop('Sex', axis=1)
teste = teste.drop('Sex', axis=1)

In [10]:
# Importando o RobustScaler
from sklearn.preprocessing import RobustScaler

# Criando o scaler
transformer = RobustScaler().fit(treino[['Age', 'Fare']])

# Fazendo a transformação dos dados
treino[['Age', 'Fare']] = transformer.transform(treino[['Age', 'Fare']])

# Fazendo o mesmo para a base de teste
transformer = RobustScaler().fit(teste[['Age', 'Fare']])
teste[['Age', 'Fare']] = transformer.transform(teste[['Age', 'Fare']])

In [11]:
# Criando uma função para verificar se dois valores são vazios
def sozinho(a,b):
    if(a==0 and b==0):
        return 1
    else:
        return 0

# Aplicando essa função na base de treino
treino['Sozinho'] = treino.apply(lambda x: sozinho(x.SibSp, x.Parch), axis=1)
# Fazendo o mesmo para base de teste
teste['Sozinho'] = teste.apply(lambda x: sozinho(x.SibSp, x.Parch), axis=1)

In [12]:
# Criando uma nova coluna com o total de familiares a bordo
treino['Familiares'] = treino.SibSp + treino.Parch
teste['Familiares'] = teste.SibSp + teste.Parch

In [13]:
# Usando o OrdinalEncoder para a coluna Embarked
# Importando OrdinalEncoder
from sklearn.preprocessing import OrdinalEncoder
categorias = ['S', 'C', 'Q']

enc = OrdinalEncoder(categories=[categorias], dtype='int32')
# faznedo o fit com os dados
enc = enc.fit(treino[['Embarked']])
# Adicionando essa coluna na base de treino original
treino['Embarked'] = enc.transform(treino[['Embarked']])

# Adicionando na base de teste original
teste['Embarked'] = enc.transform(teste[['Embarked']])

In [14]:
# Visualizando a base de treino
treino.head(3)

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,Embarked,MaleCheck,Sozinho,Familiares
0,1,0,3,-0.592240,1,0,-0.312011,0,1,0,1
1,2,1,1,0.638529,1,0,2.461242,1,0,0,1
2,3,1,3,-0.284548,0,0,-0.282777,0,0,1,0


## Reutilizando os modelos da parte 4
**Observação:** Além de usar o train_test_split, vou usar o grid_search

In [15]:
# Importando o train_test_split
from sklearn.model_selection import train_test_split

In [16]:
# Sepearando a base de treino em X e Y
X =  treino.drop(['PassengerId', 'Survived'], axis=1)
y = treino.Survived

In [17]:
# Separando em treino e validação
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

### Regressão Logística

In [18]:
# Importando
from sklearn.linear_model import LogisticRegression

In [19]:
# Criando o classificador
clf_rl = LogisticRegression(random_state=42)

In [20]:
# Definindo os parametros
parametros_rl = {
    'penalty': ['l1','l2'],
    'C': [0.01, 0.1, 1, 10],
    'solver': ['lbfgs', 'liblinear', 'saga'],
    'max_iter': [100, 1000, 5000, 10000]
}

### RandomForest

In [21]:
# Importação
from sklearn.ensemble import RandomForestClassifier

In [22]:
# Criando o classificador
clf_rf = RandomForestClassifier(random_state=42)

In [23]:
# Definindo parametros
parametros_rf = {
    "n_estimators": [100, 200, 500, 1000],
    "criterion": ["gini", "entropy", "log_loss"],
    "max_depth": [2, 4, 6, 8, None],
    "max_features": ["sqrt", "log2", None]
}

### MLPClassifier (Redes Neurais)

In [24]:
# Importando
from sklearn.neural_network import MLPClassifier

In [25]:
# Criando o classificador
clf_mlp = MLPClassifier(random_state=42)

In [26]:
# Definindo os parâmetros
parametros_mlp = {
    "solver": ["lbfgs", "sgd", "adam"],
    "alpha": [1e-1, 1e-3, 1e-5, 1e-7, 1e-10],
    "max_iter": [200, 500, 1000, 5000]
}

## Usando o Grid_search

In [27]:
# Ignorando os avisos
import warnings 
warnings.filterwarnings('ignore')

In [28]:
# Importando o datetime para visualizar a hora atual
from datetime import datetime
def hora_atual():
    agora = datetime.now()
    print(str(agora.hour)+':'+str(agora.minute)+':'+str(agora.second))

In [29]:
hora_atual()

10:31:30


In [30]:
# Importando o KFold e GridSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold

In [31]:
# Para a Regressão Logistica
hora_atual()
kfold_rl = KFold(shuffle=True, random_state=42, n_splits=8)
grid_search_rl = GridSearchCV(clf_rl, parametros_rl, scoring='accuracy', cv=kfold_rl)
grid_search_rl = grid_search_rl.fit(X_train, y_train)
hora_atual()

10:31:30
10:31:40


In [32]:
# Para o RandomForest
hora_atual()
kfold_rf = KFold(shuffle=True, random_state=42, n_splits=8)
grid_search_rf = GridSearchCV(clf_rf, parametros_rf, scoring='accuracy', cv=kfold_rf)
grid_search_rf = grid_search_rf.fit(X_train, y_train)
hora_atual()

10:31:40
10:50:37


In [33]:
# Para o MLPClassifier
hora_atual()
kfold_mlp = KFold(shuffle=True, random_state=42, n_splits=8)
grid_search_mlp = GridSearchCV(clf_mlp, parametros_mlp, scoring='accuracy', cv=kfold_mlp)
grid_search_mlp = grid_search_mlp.fit(X_train, y_train)
hora_atual()

10:50:37
10:59:47


## Verificando os melhores scores

In [38]:
# Verificando o melhor score da regressão logística
grid_search_rl.best_score_

np.float64(0.8089887640449438)

In [39]:
# Para o RandomForest
grid_search_rf.best_score_

np.float64(0.8314606741573034)

In [40]:
# Para o MPLClassifier
grid_search_mlp.best_score_

np.float64(0.8174157303370786)

## Verificando os melhores parâmetros

In [41]:
# Verificando os melhores parametros da Regressão Logística
grid_search_rl.best_params_

{'C': 0.1, 'max_iter': 100, 'penalty': 'l2', 'solver': 'lbfgs'}

In [42]:
# Para o RandomForest
grid_search_rf.best_params_

{'criterion': 'entropy',
 'max_depth': 6,
 'max_features': 'sqrt',
 'n_estimators': 100}

In [43]:
# Para o MPLClassifier
grid_search_mlp.best_params_

{'alpha': 0.1, 'max_iter': 200, 'solver': 'adam'}

## Fazendo a previsão nos dados de validação com cada um dos melhores modelos

In [44]:
# Para Regressão Logística
clf_best_rl = grid_search_rl.best_estimator_
y_pred_rl = clf_best_rl.predict(X_val)

In [45]:
# Para RandomForest
clf_best_rf = grid_search_rf.best_estimator_
y_pred_rf = clf_best_rf.predict(X_val)

In [46]:
# Para o MPLClassifier
clf_best_mlp = grid_search_mlp.best_estimator_
y_pred_mlp = clf_best_mlp.predict(X_val)

## Avaliando a acurácia

In [48]:
# Importando
from sklearn.metrics import accuracy_score

In [50]:
# Para a regressão logística
accuracy_score(y_val, y_pred_rl)

0.8044692737430168

In [52]:
# Para o Random Forest
accuracy_score(y_val, y_pred_rf)

0.8100558659217877

In [53]:
# Para o MLPClassifier
accuracy_score(y_val, y_pred_mlp)

0.8100558659217877

## Avaliando a matriz de confusão

In [54]:
# Importando 
from sklearn.metrics import confusion_matrix

In [56]:
# Para a Regressão Logisitca
confusion_matrix(y_val, y_pred_rl)

array([[91, 14],
       [21, 53]])

In [57]:
# Para o Random Forest
confusion_matrix(y_val, y_pred_rf)

array([[93, 12],
       [22, 52]])

In [58]:
# Para o MPLClassifier
confusion_matrix(y_val, y_pred_mlp)

array([[92, 13],
       [21, 53]])

## Fazendo a previsão para os dados de teste

In [59]:
# Visualizando o X_train
X_train.head(3)

,Pclass,Age,SibSp,Parch,Fare,Embarked,MaleCheck,Sozinho,Familiares
331,1,1.215452,0,0,0.608317,0,1,1,0
733,2,-0.515317,0,0,-0.062981,0,1,1,0
382,3,0.176991,0,0,-0.282777,0,1,1,0


In [60]:
# Visualizando a base de teste
teste.head(3)

,PassengerId,Pclass,Age,SibSp,Parch,Fare,Embarked,MaleCheck,Sozinho,Familiares
0,892,3,0.331562,0,0,-0.280670,2,1,1,0
1,893,3,1.311954,1,0,-0.315800,0,0,0,1
2,894,2,2.488424,0,0,-0.201943,2,1,1,0


In [61]:
# Para a base de teste ser igual a base de treino, precisamos eliminar a coluna de id
X_teste = teste.drop('PassengerId', axis=1)

In [62]:
# Utilizando o melhor modelo
y_pred = clf_best_rf.predict(X_teste)

In [63]:
# Criando uma nova coluna com a previsão na base de teste
teste['Survived'] = y_pred

In [64]:
# Selecionando apenas a coluna de Id e Survived para fazer o envio
base_envio = teste[['PassengerId', 'Survived']]

In [65]:
base_envio.to_csv('resultados5.csv', index=False)